# UA-DETRAC 微调 YOLOv8n（单类 vehicle）

**用法**：把 `data/dataset/` 打包传成 Kaggle Dataset，在右边栏 **Add Input** 挂上，然后从上往下依次运行。

**跑之前检查两件事**（缺一个就跑不了）：
1. 右边栏 **Settings → Accelerator** 选 `GPU T4 x2`
2. 右边栏 **Settings → Internet** 打开

预计耗时：50 轮大约 1 小时。

In [ ]:
# 装 ultralytics。Kaggle 的 PyTorch 和 CUDA 是现成的，只缺这个
!pip install ultralytics -q

In [ ]:
import os, glob, shutil, random, re

# 找到挂上来的数据集（里面有 images/ 和 labels/）
cands = [d for d in glob.glob('/kaggle/input/*') if os.path.isdir(os.path.join(d, 'images'))]
if not cands:
    cands = [d for d in glob.glob('/kaggle/input/*/*') if os.path.isdir(os.path.join(d, 'images'))]
assert cands, '没找到数据集。检查右边栏 Add Input 挂上了没有'
SRC = cands[0]
print('数据在：', SRC)
print('图片数：', len(glob.glob(os.path.join(SRC, 'images', '*.jpg'))))

In [ ]:
# /kaggle/input 是只读的，先拷到可写的地方
WORK = '/kaggle/working/dataset'
os.makedirs(WORK, exist_ok=True)
for sub in ('images', 'labels'):
    src, dst = os.path.join(SRC, sub), os.path.join(WORK, sub)
    if not os.path.exists(dst):
        shutil.copytree(src, dst)
        print('拷好', sub)
    else:
        print(sub, '已存在，跳过')

with open(os.path.join(WORK, 'classes.txt'), 'w') as f:
    f.write('vehicle\n')

print('\n工作目录：', WORK)

In [ ]:
# 重新生成 train/val 列表。
# 本地那份写的是 Windows 路径，到这儿没用，必须重生成。
# 下面这段和本地 tools/split_dataset.py 逻辑一样，种子也固定，所以划分结果和本地完全相同：
# 按【摄像头】分家，不是按图片随机分，避免同一机位的图两边都出现导致分数虚高。

def group_of(name):
    m = re.match(r'^(.*?)_[A-Za-z]*\d+$', name)
    return m.group(1) if m else name

imgs = sorted(glob.glob(os.path.join(WORK, 'images', '*.jpg')))
groups = {}
for p in imgs:
    groups.setdefault(group_of(os.path.splitext(os.path.basename(p))[0]), []).append(p)

names = sorted(groups)
random.seed(0)          # 种子固定，划分结果和本地一致
random.shuffle(names)
n_val = max(1, int(round(len(names) * 0.2)))
n_val = min(n_val, len(names) - 1)
val_groups = set(names[:n_val])

tr = sorted(p for g, ps in groups.items() if g not in val_groups for p in ps)
va = sorted(p for g, ps in groups.items() if g in val_groups for p in ps)

for path, items in (('train.txt', tr), ('val.txt', va)):
    with open(os.path.join(WORK, path), 'w') as f:
        f.write('\n'.join(items) + '\n')

with open(os.path.join(WORK, 'data.yaml'), 'w') as f:
    f.write('path: %s\ntrain: train.txt\nval: val.txt\nnc: 1\nnames: [vehicle]\n' % WORK)

print('训练集 %d 张（%d 个摄像头）' % (len(tr), len(names) - len(val_groups)))
print('验证集 %d 张（%d 个摄像头）' % (len(va), len(val_groups)))
print('验证集摄像头：', ', '.join(sorted(val_groups)))

In [ ]:
# 开训。
# imgsz=640：UA-DETRAC 原图才 960x540，车够大，640 完全够，而且比 960 快 2.25 倍。
#             以后推理也必须用 640，两边不一致精度会掉。
# epochs=50： 微调不需要 100 轮。
# batch=16：  T4 16G 显存扛得住；报显存不够就改 8。
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
model.train(
    data=os.path.join(WORK, 'data.yaml'),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    degrees=5.0,      # 监控机位固定，不用转太多
    scale=0.5,        # 远处车和近处车大小差别大，这个重要
    mosaic=1.0,       # 四图拼接，增加小目标样本
    workers=2,
    project='/kaggle/working',
    name='detrac_vehicle_v1',
)

In [ ]:
# 看结果。best.pt 就是要下载回去的那份。
import pandas as pd
from IPython.display import Image, display

RUN = '/kaggle/working/detrac_vehicle_v1'
df = pd.read_csv(os.path.join(RUN, 'results.csv'))
df.columns = [c.strip() for c in df.columns]
cols = [c for c in df.columns if 'mAP50' in c or c == 'epoch']
print(df[cols].tail(10).to_string(index=False))

for name in ('results.png', 'confusion_matrix.png'):
    p = os.path.join(RUN, name)
    if os.path.exists(p):
        display(Image(p))

print('\n模型在这：%s/weights/best.pt' % RUN)
print('大小：%.1f MB' % (os.path.getsize(os.path.join(RUN, 'weights', 'best.pt')) / 1e6))

## 跑完之后

1. 右上角 **Save Version → Save & Run All**，让它在后台把结果固定下来
2. 跑完在 **Output** 标签页里下载 `detrac_vehicle_v1/weights/best.pt`
3. 下回来放到本地的 `D:\multi-road-crossing-vehicle-count\models\` 下
4. 把 `01`~`04` 脚本里的 `MODEL` 改成新模型的路径，再跑一遍那 4 个测试视频，看漏检少没少